# Pandas for Data Analysis

ExxonMobil, 10/30/17

# Recap

Numpy
* Package for fast numeric computation in Python.
* Interacts with other packages nicely
* Pandas is built on top of numpy

In [ ]:
import numpy as np
# A quick example showing the speedup reached with numpy computations
arr, ls = np.arange(30000), range(30000)
%timeit arr*10
%timeit [i*10 for i in ls]

# Today

**Outline:**
* Pandas data types
* File I/O
* Working with DataFrames
    * Selecting and Querying
    * Functions
    * Plotting
    * Joins 
    * Group By

# Pandas data types

In [ ]:
# this is the standard import for pandas
import pandas as pd

## Series

** Series: ** A series is a Numpy array with axis labels. The label is called an "index" and "names" each element.

In [ ]:
# pd.Series will create a series. If no index is given it defaults to the len(data)
print('No index \n--------')
print(pd.Series(np.arange(5)), '\n')

print('Index \n--------')
print(pd.Series(np.arange(5), index = ['zero','one','two','three','four']))

## DataFrames

**DataFrame:** A dataframe is a tabular data structure with rows and columns. Rows and columns have names. Rows are called the index and columns are called columns. 

<img src="images/dataframe.png">

In [ ]:
# We can create a dataframe from a Python dictionary
# The keys will be the column names
# The values will be the values in the cells
# The index (row name) defaults to the row number starting at 0
name_city_dictionary = {'name':['Mike', 'Andrew'], 'city':['NYC', 'SF']}
df = pd.DataFrame(name_city_dictionary)
df.head()

In [ ]:
# We can change the index (row name) by using the index parameter as creation
name_city_dictionary = {'name':['Mike', 'Andrew'], 'city':['NYC', 'SF']}
df = pd.DataFrame(name_city_dictionary, index=['person1', 'person2'])
df.head()

In [ ]:
# To get the column and row names using .columns and .index
print('Columns')
print(list(df.columns), '\n')

print('Index ')
print(list(df.index), '\n')

# Use .values to access the values in the cells. These are numpy arrays
print('Values are numpy arrays')
print(type(df.values))
print(df.values)

### Main Points
* Pandas has two primary data structures
    * Series: numpy array with row labels
    * DataFrame: Table with rows and columns
* A DataFrame can be created from a Python dictionary
* df.index to get row names
* df.columns to get column names
* df.values to get DataFrame values as a numpy array

## File I/O

Pandas has made is simple to read different files into a dataframe for analysis 

We'll use an energy usage dataset from the U.S. Energy Information Administration as an example. We've downloaded the "Energy consumption estimates by sector, 1949– 2012". It's in the data folder

https://www.eia.gov/totalenergy/data/annual/#consumption.



In [ ]:
# .read_csv() takes the path to a csv file and returns a DataFrame
df_csv = pd.read_csv('data/MER_T02_01.csv')
df_csv.head()

In [ ]:
# EnergyData.xlsx is a spreadsheet containing the above energy data
# There are two sheets (before2000, after2000) 
# before2000: has the data from before the year 2000
# after2000: has the data from after the year 2000
# By default read_excel only returns the first sheet
df_excel = pd.read_excel('data/EnergyData.xlsx')
df_excel.head()

In [ ]:
# the parameter "sheetname" allows us to specify which sheet to read in
df_excel_after2000 = pd.read_excel('data/EnergyData.xlsx', sheetname = 'after2000')
df_excel_after2000.head()

In [ ]:
# We can read in multiple sheets at the same time
# A dictionary will be returned where the key is the sheet name and value is the dataframe

# The list of sheet names can be passed the sheetname or by passing None all sheets will be returned
df_excel_all = pd.read_excel('data/EnergyData.xlsx', sheetname = None)
print(type(df_excel_all))
print(df_excel_all.keys())
df_excel_all['before2000'].head()

In [ ]:
# delete dataframes using del
del df_csv
del df_excel
del df_excel_after2000
del df_excel_all

There are options to read in data from HTML, SQL, HDF5, and many others. More options can be found [here](https://pandas.pydata.org/pandas-docs/stable/io.html])

In [ ]:
import sqlite3 as sq3

path = '../../sql'
query = 'CREATE TABLE instructors (Name varchar(255), City varchar(255))'
con = sq3.Connection(path + 'instructors.db')
con.execute(query)
data = np.array([['Mike','Andrew'],['NYC','SF']])
con.executemany('INSERT INTO instructors VALUES (?, ?)', data)
con.commit()

In [ ]:
import sqlite3 as sq3
import pandas.io.sql as pds

# We'll cover databases more next week but here's an example of querying
# the instructors table from a sqlite database engine
con = sq3.Connection(path + 'instructors.db')
instructors = pds.read_sql('SELECT * FROM instructors', con)
instructors.head()

In [ ]:
# We can similarly write DataFrames to files using similar commands as reading in files
instructors.to_csv("data/instructors.csv")

### Main Points
There are many options to read and write data. A few common examples are:
* **read_excel:** read from excel files
* **read_csv:** read from csv files
* **to_csv:** write to csv files

More options can be found [here](https://pandas.pydata.org/pandas-docs/stable/io.html])

### Exercise

Read in the census income dataset in the data folder to a DataFrame called census_income. What are the column names. View the first 5 rows

# Working with DataFrames

## Selecting rows and columns and querying DataFrames

A DataFrame can be thought as a Python dictionary with the column names as keys and each column is a Series. You can select columns from a DataFrame using brackets

In [ ]:
print(type(df['name']))
df['name']

Rows can be selected in many different ways:
* loc: index labels
* iloc: row positions 
* ix: Both position and label. Acts like loc unless the the label isn't in the index 

In [ ]:
print('select by label using loc')
print(df.loc['person2'], '\n')

print('select by position using iloc')
print(df.iloc[0], '\n')

In [ ]:
# .query() selects columns using a boolean
print(df.query("name == 'Mike'"))

### Main Points

* Select columns with brackets. df[column_name]
* Select rows:
    * **loc:** index labels
    * **iloc:** row positions 
* **.query():** query columns with a boolean expression

### Assignment

What are the unique education values in the census dataset. Select just people with Bachelors degrees and show the first 5 rows

## DataFrame Functions

In [ ]:
df = pd.read_csv('data/MER_T02_01.csv')
df.head()

In [ ]:
# .apply() will apply an input function to every column or row. 
# axis = 0 or ‘index’: apply function to each column
# axis = 1 or ‘columns’: apply function to each row
def year_number(row):
    return str(row['YYYYMM'])[0:4]

df.head().apply(year_number, axis=1)

In [ ]:
# We can add a column directly 
df['Year'] = df.apply(year_number, axis=1)
df.head()

In [ ]:
# .dtypes returns the types of the data
# Notice that Year is of type object
print(df.dtypes)

In [ ]:
# We can use .to_datetime() to convert to a datetime type
df['Year'] = pd.to_datetime(df['Year'])
print(df.dtypes)

More on .to_datetime() can be found [here](http://pandas.pydata.org/pandas-docs/version/0.20/generated/pandas.to_datetime.html)

More on Python datetimes can be found [here](https://docs.python.org/3/library/datetime.html)

In [ ]:
# .describe() gives descriptive statistics on the DataFrame
df.describe()

In [ ]:
# .value_counts() give the distribution of categorical variables
instructors = pd.DataFrame({'name':['Andrew','Mike', 'Julia'], 'city':['SF','NYC','NYC']})
instructors['city'].value_counts()

In [ ]:
# .append() appends rows to a DataFrame. New columns are added with nan values
new_instructor = pd.DataFrame({'name':['Seth', np.nan], 'city':['CH', np.nan], 'role':['SDS',np.nan]})
instructors = instructors.append(new_instructor)
instructors

Pandas DataFrames can handle missing data. Since they are built on top of Numpy arrays, missing values have the "Numpy not a number" data type by default, which will show up as "NaN".

In [ ]:
# pd.isnull() detects missing values in a DataFrame
pd.isnull(instructors)

In [ ]:
# .dropna() drops rows with na values
# how = "any" drops any row with an na value
# how = "all" drops only rows with all na values
print(instructors.dropna(how="any"), '\n')
print(instructors.dropna(how="all"))

### Main Points
Common functions: 
* **.apply():** Applies a function to every row or column
* **.describe():** Get descriptive statistics
* **.value_counts():** Distribution of categorical variables
* **.append():** Append rows to the end of a DataFrame
* **.isnull():** Detect missing values
* **.dropna():** Drop rows with missing values. Options for any or all rows

### Assignment

What is the distribution of eduation types? Are there any missing age values?


## Plotting

In [ ]:
# to plot in pandas select the column followed by the plot type
# .hist() is you would create a histogram
%matplotlib inline
new_df = df.query('Value < 20000')
new_df['Value'].hist(bins = 20)

### Main Points

Pandas offers many other kinds of plots "out of the box", such as:

* `.bar()` for bar plots
* `.hist()` for histograms
* `.scatter()` for scatter plots

See [here](https://pandas.pydata.org/pandas-docs/stable/visualization.html#other-plots) for more details.

### Assignment

Create a histogram of the ages of just the men

## Joins

Pandas DataFrames can be joined with similar syntax to that which exists in SQL. We can join DataFrames using inner, right, left, or outer joins, just as in SQL. 

In [ ]:
df1 = pd.DataFrame({'A': ['A0', 'A1', 'A2', 'A3'], 
                    'B': ['B0', 'B1', 'B2', 'B3'],
                    'C': ['C0', 'C1', 'C2', 'C3'], 
                    'D': ['D0', 'D1', 'D2', 'D3']}, 
                   index=[0, 1, 2, 3])

df4 = pd.DataFrame({'E': ['E2', 'E3', 'E6', 'E7'],
                    'F': ['F2', 'F3', 'F6', 'F7'],
                    'G': ['G2', 'G3', 'G6', 'G7']},
                   index=[2, 3, 6, 7])

In [ ]:
# Let's say we wanted to do an inner join on the following two dataframes
print(df1, '\n')
print(df4)

In [ ]:
# we can do this with .join()
df1.join(df4, 
         how="inner")

**Note:** Joins by default happen based on the DataFrames' indices, though this can be changed using the `on` keyword argument.

More on doing joins in Pandas can be found [here](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.join.html)

## Group by and aggregation

Pandas also has "group by" functionality that mimicks SQL's. There is thorough documentation along with simple examples on how to do group by [here](https://pandas.pydata.org/pandas-docs/stable/groupby.html).

In [ ]:
df_gb_example = pd.DataFrame({'Name':['Mike', 'Andrew', 'Mike'], 'Amount':[1.00, 2.00,3.00]})
df_gb_example.groupby('Name').sum()

# More Reading and Exercises

## Copy vs. Deep Copy

In [ ]:
df2 = pd.DataFrame({ 'A' : 1., 
                     'B' : pd.Timestamp('20130102'),
                     'C' : pd.Series(1,index=list(range(4)),dtype='float32'),
                     'D' : np.array([3] * 4,dtype='int32'),
                     'E' : pd.Categorical(["test","train","test","train"]),
                     'F' : 'foo' })
df2

Because Pandas is built on top of Numpy, it mimics Numpy's copying behavior as well. 

In [ ]:
df3 = df2
df3['A'][0] = 2.0
df2

Notice that the 0th index of column A changed in df2 even though we changed df3. The proper way to copy in Pandas is to use `deep copy`, as in Numpy:

In [ ]:
# Reset df2
df2 = pd.DataFrame({ 'A' : 1., 
                     'B' : pd.Timestamp('20130102'),
                     'C' : pd.Series(1,index=list(range(4)),dtype='float32'),
                     'D' : np.array([3] * 4,dtype='int32'),
                     'E' : pd.Categorical(["test","train","test","train"]),
                     'F' : 'foo' })

df3 = df2.copy(deep=True)
df3['A'][0] = 2.0
df2

Now, we get the copying behavior we want: changing `df3` does not affect `df2`.

### Exercise

Create and add a "month" column to the below DataFrame. Do you think you could convert this to a `datetime`? Why or why not?

In [ ]:
df = pd.read_csv('data/MER_T02_01.csv')
df.head()

### Question

reassign df so it ignores the rows where the Month is 13. 

### Question

Write a function that returns a datetime from input like the column YYYYMM. Use this function to create a new column for df called datetime. See if you can view the datetime documention [here](https://docs.python.org/3/library/datetime.html)

### Question

What are the different values of the Description column?

### Question:

Now that you know the different values of the `Description` column: select only the rows in the DataFrame where that column is equal to "Total Energy Consumed by the Industrial Sector". Save the result in a DataFrame called `industrial`. Use the `query` syntax from above and be careful of the double and single quotes!

## Pickling

One problem with reading and writing from CSVs is that the data types of your columns may not be preserved. Python has a way around this that works with Python objects in general: "pickling". If we write a DataFrame to a pickle object, it will preserve its data types and everything:

In [ ]:
industrial.to_pickle("industrial.p")

In [ ]:
industrial_new = pd.read_pickle("industrial.p")

In [ ]:
industrial_new.dtypes

Note: if we had read this is from a CSV file, the type of the `datetime` column would be a string or `object` ("object" is just the type Pandas assigns to things when it doesn't know what type they are). 

**Let's pause and reflect:** We have now:

1. Read in data from a CSV (or pickle file)
2. Modified it
3. Outputted it back out to a CSV (or pickle file)

This basic process is how you can automate many tasks you would otherwise have to open up Excel and do manually using "point-and-click" methods. 

### Question

Create a line plot of the values for the industrial DataFrame

### Question

Nice plot but what's wrong with the x-axis? Change the index of the industrial DataFrame and try recreating the plot